# Fine-tune the Tunisian Arabizi sentiment model (Model B)

Run this on a **free GPU** (Colab: Runtime -> Change runtime type -> T4 GPU; or Kaggle). The base model is small (~270M params), so a T4 trains it in minutes.

It fine-tunes `cardiffnlp/twitter-xlm-roberta-base` on a Tunisian Arabizi sentiment corpus (TUNIZI), then reports per-language macro-F1 and saves the model + a model card. Download the saved folder and mount it into the app (`ARABIZI_MODEL=/models/arabizi`) to route Arabizi comments to it.

## 1. Get the code and install the training extras

In [ ]:
# Replace OWNER with your GitHub username.
!git clone https://github.com/OWNER/community-management.git
%cd community-management
!pip install -q -e ".[nlp,train]"

## 2. Provide the TUNIZI data

Either point `--dataset` at a HuggingFace TUNIZI mirror, or upload a CSV with `text` and `label` columns (labels can be positive/negative/neutral, or 1/0, etc. — they get normalized). To upload in Colab:

In [ ]:
# Optional: upload a local tunizi.csv (skip if using --dataset)
from google.colab import files  # noqa
uploaded = files.upload()  # choose your tunizi.csv

## 3. Fine-tune (stratified 70/10/20, lr 2e-5, early stopping on macro-F1)

In [ ]:
!python -m app.nlp.training.finetune_tunizi \
    --csv tunizi.csv \
    --base-model cardiffnlp/twitter-xlm-roberta-base \
    --output-dir models/arabizi \
    --epochs 4
# (or swap --csv tunizi.csv for --dataset <hf_id>)

## 4. Before/after: evaluate Model A vs the fine-tuned Model B

Run the same held-out set through the multilingual baseline and your fine-tuned model to get the per-language macro-F1 table. The delta on `aeb-latn` is the headline result for the report.

In [ ]:
print('=== Model A (baseline) ===')
!python -m app.nlp.training.evaluate --csv tunizi.csv --model cardiffnlp/twitter-xlm-roberta-base-sentiment
print('\n=== Model B (fine-tuned) ===')
!python -m app.nlp.training.evaluate --csv tunizi.csv --model models/arabizi

## 5. Download the trained model


In [ ]:
!zip -r arabizi_model.zip models/arabizi
from google.colab import files  # noqa
files.download('arabizi_model.zip')
# Unzip into a folder your Docker api/worker can see, then set ARABIZI_MODEL to that path.